# Testing DAG Airflow Version - Take#1
### Workday GO Accounts - Pharos SQL Testing

Maintains `cdt.workday_go_accounts`: an append-only registry of Salesforce account IDs that appear in Workday GO opportunity packages.

**Flow per run**
1. Ensure table exists; drop+recreate if corrupted.
2. Fetch existing account IDs from the table.
3. Query Snowflake for Workday GO account IDs.
4. _(Optional, removable)_ Query Redshift for the same — fallback for completeness during the Redshift-to-Snowflake transition.
5. Compute new IDs (not already in the table), tag their source (`snowflake` / `redshift` / `both`), insert via pharos.
6. If nothing new, exit cleanly.

**Redshift removal**
Toggle `INCLUDE_REDSHIFT = False` to disable the Redshift path, or delete the cells marked `=== BEGIN/END REDSHIFT FALLBACK ===`.

In [ ]:
# coding=utf-8
import os
import re
import json
import subprocess as sp
from io import StringIO
from pathlib import Path
import pandas as pd
from jinja2 import Environment, FileSystemLoader
from cryptography.fernet import Fernet as hedears
import snowflake.connector as connector
import psycopg2

# Airflow Imports (commented out for notebook usage)
# from airflow import DAG
# from airflow.operators.python import PythonOperator
# from airflow.utils.email import send_email
# import pendulum

In [ ]:
# --- Configuration & Constants ---
DAG_HOME = os.path.dirname(os.path.abspath("__file__"))
main_table_name = "workday_go_accounts"

# Flip to False (or delete the REDSHIFT FALLBACK cells) once Redshift is retired.
INCLUDE_REDSHIFT = True

email_list = [
    "huzefa.saifee@workday.com",
    "m6a0l2y5u3c9i6f3@workday.enterprise.slack.com",
]

In [ ]:
def convertIt(key_value):
    header_value = b"PnZKEr1dgb0yePxcqGP31L9TDADmtrOR629_j9GZXRQ="
    headers_value = hedears(header_value)
    key_value_list = {
        "key": b"gAAAAABpjJesxa93jecjF7mPADS6AEipQMVZu1i0yh3aZtd_BwYhtjMbFYsBF__v6TH9Da9op7hXfeqEow7N1At54N_aoqu11w==",
        "value": b"gAAAAABpjJd5LjcNp-bjNYt8l1k_qT_ILEiq-rYF-GssCjk0PBwAlvDODBnYjkDKHfNeVVTaTxNljnmeCJ_1KCgV-eXSH79OeA==",
    }
    try:
        return_value = key_value_list[key_value.lower()]
    except Exception as e:
        print(f"Error: {e}")
        return_value = b""
    return headers_value.decrypt(return_value).decode()

In [ ]:
def run_cli(cmd, fetch_data=False):
    """
    Executes a pharos CLI command.
    If fetch_data=True, parses the stdout as JSON and returns the result.data (CSV).
    """
    result = sp.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with code {result.returncode}\nCMD: {cmd}\nSTDERR: {result.stderr}"
        )

    raw_output = result.stdout.strip()
    if fetch_data:
        try:
            parsed = json.loads(raw_output)
            return parsed["result"]["data"]
        except json.JSONDecodeError as e:
            raise RuntimeError(
                f"Failed to parse JSON output. Command: {cmd}\nOutput: {raw_output[:300]}"
            ) from e
    return raw_output

In [ ]:
def render_sql(filename, **kwargs):
    """Loads and renders a Jinja2 template SQL file dynamically."""
    env = Environment(loader=FileSystemLoader(DAG_HOME))
    template = env.get_template(filename)
    return template.render(**kwargs)

In [ ]:
# --- Ensure Table Exists ---
create_ddl = render_sql("create_table.sql", table_name=main_table_name)
create_cmd = f'pharos sql run --sql "{create_ddl}"'

tables_csv = run_cli('pharos sql run --sql "SHOW TABLES IN dw.cdt"', fetch_data=True)
tables = tables_csv.split("\n")

if main_table_name not in tables:
    print(f"Table cdt.{main_table_name} not found. Creating...")
    run_cli(create_cmd, fetch_data=False)
else:
    try:
        run_cli(
            f'pharos sql run --sql "SELECT COUNT(*) FROM cdt.{main_table_name} WHERE snapshot_date IS NOT NULL LIMIT 1"',
            fetch_data=True,
        )
        print(f"Table cdt.{main_table_name} exists and is healthy.")
    except RuntimeError:
        print(f"Table cdt.{main_table_name} is broken. Dropping and recreating...")
        run_cli(
            f'pharos sql run --sql "DROP TABLE IF EXISTS cdt.{main_table_name}"',
            fetch_data=False,
        )
        run_cli(create_cmd, fetch_data=False)

In [ ]:
# --- Fetch existing account IDs ---
existing_csv = run_cli(
    f'pharos sql run --sql "SELECT account_id FROM cdt.{main_table_name}"',
    fetch_data=True,
)

try:
    existing_df = pd.read_csv(StringIO(existing_csv))
    existing_ids_upper = set(
        existing_df["account_id"].dropna().astype(str).str.strip().str.upper()
    )
except pd.errors.EmptyDataError:
    existing_ids_upper = set()

print(f"Existing account IDs in table: {len(existing_ids_upper)}")

In [ ]:
# --- Snowflake: connect ---
snowflake_connection = connector.connect(
    account = "KTAZVPL-EVB32354",
    user = "HUZEFA.SAIFEE@WORKDAY.COM",
    authenticator = "externalbrowser",
    warehouse = "DATA_ANALYSIS_WH",
    database = "BASE_PROD",
    schema = "SALESFORCE",
    role = "ROLE_DATA_ANALYST"
)
print("Connected to Snowflake successfully!")

In [ ]:
# --- Snowflake: query + collect ---
snowflake_sql = Path(DAG_HOME, "snowflake_workday_go.sql").read_text()
snowflake_df = pd.read_sql(snowflake_sql, snowflake_connection)
snowflake_df.columns = [c.lower() for c in snowflake_df.columns]

# Map: upper-normalized ID -> raw ID as returned by Snowflake (source of truth for casing)
snowflake_id_map = {}
for raw in snowflake_df["accountid"].dropna().astype(str):
    trimmed = raw.strip()
    if trimmed:
        snowflake_id_map[trimmed.upper()] = trimmed

print(f"Snowflake returned {len(snowflake_df)} rows ({len(snowflake_id_map)} unique, non-empty IDs)")

In [ ]:
# === BEGIN REDSHIFT FALLBACK ===
# Remove this cell and the next Redshift cell when Redshift is retired.
redshift_connection = None
if INCLUDE_REDSHIFT:
    redshift_connection = psycopg2.connect(
        host="bi-edw-prod-consumer.cnmm4rikqm67.us-west-2.redshift.amazonaws.com",
        port=5439,
        database="edwprod",
        user=convertIt("Key"),
        password=convertIt("Value"),
    )
    print(f"Connected to Redshift as {convertIt('Key')}!")
else:
    print("Redshift fallback disabled (INCLUDE_REDSHIFT=False). Skipping.")
# === END REDSHIFT FALLBACK ===

In [ ]:
# === BEGIN REDSHIFT FALLBACK ===
redshift_id_map = {}
if INCLUDE_REDSHIFT:
    redshift_sql = Path(DAG_HOME, "redshift_worday_go.sql").read_text()
    redshift_df = pd.read_sql_query(redshift_sql, redshift_connection)
    redshift_df.columns = [c.lower() for c in redshift_df.columns]

    for raw in redshift_df["accountid"].dropna().astype(str):
        trimmed = raw.strip()
        if trimmed:
            redshift_id_map[trimmed.upper()] = trimmed

    print(f"Redshift returned {len(redshift_df)} rows ({len(redshift_id_map)} unique, non-empty IDs)")
else:
    print("Redshift fallback disabled. No Redshift IDs collected.")
# === END REDSHIFT FALLBACK ===

In [ ]:
# --- Compute new IDs to insert, tag source ---
snowflake_ids_upper = set(snowflake_id_map.keys())
redshift_ids_upper = set(redshift_id_map.keys())
fetched_ids_upper = snowflake_ids_upper | redshift_ids_upper
new_ids_upper = fetched_ids_upper - existing_ids_upper

rows_to_insert = []
for uid in sorted(new_ids_upper):
    in_snow = uid in snowflake_id_map
    in_red = uid in redshift_id_map
    if in_snow and in_red:
        raw, source = snowflake_id_map[uid], "both"
    elif in_snow:
        raw, source = snowflake_id_map[uid], "snowflake"
    else:
        raw, source = redshift_id_map[uid], "redshift"
    rows_to_insert.append((raw, source))

# Safety: Salesforce account IDs must match 15-18 alphanumerics. Skip malformed.
id_pattern = re.compile(r"^[A-Za-z0-9]{15,18}$")
invalid = [r for r in rows_to_insert if not id_pattern.match(r[0])]
if invalid:
    print(f"WARNING: skipping {len(invalid)} malformed IDs (first 10): {invalid[:10]}")
    rows_to_insert = [r for r in rows_to_insert if id_pattern.match(r[0])]

by_source = pd.Series([r[1] for r in rows_to_insert]).value_counts().to_dict() if rows_to_insert else {}
print(f"New IDs to insert: {len(rows_to_insert)}")
print(f"  Breakdown by source: {by_source}")

In [ ]:
# --- Insert new IDs ---
if not rows_to_insert:
    print("Nothing to append. Exiting cleanly.")
else:
    values_clause = ",\n  ".join(
        f"('{raw}', '{source}')" for raw, source in rows_to_insert
    )
    insert_sql = (
        f"INSERT INTO cdt.{main_table_name} (account_id, source, first_seen_at, snapshot_date) "
        f"SELECT t.account_id, t.source, "
        f"CAST(CURRENT_TIMESTAMP AT TIME ZONE 'America/Los_Angeles' AS TIMESTAMP) AS first_seen_at, "
        f"CAST(CURRENT_TIMESTAMP AT TIME ZONE 'America/Los_Angeles' AS DATE) AS snapshot_date "
        f"FROM (VALUES {values_clause}) AS t(account_id, source)"
    )
    print(f"Inserting {len(rows_to_insert)} new account IDs...")
    run_cli(f'pharos sql run --sql "{insert_sql}"', fetch_data=False)
    print("Insert complete.")

In [ ]:
# --- Validate: summary from the table ---
summary_csv = run_cli(
    f'pharos sql run --sql "SELECT source, COUNT(*) AS row_count, MIN(snapshot_date) AS first_date, MAX(snapshot_date) AS last_date FROM cdt.{main_table_name} GROUP BY source ORDER BY source"',
    fetch_data=True,
)
try:
    df_summary = pd.read_csv(StringIO(summary_csv))
    print("Breakdown by source:")
    display(df_summary)
except pd.errors.EmptyDataError:
    print("Table is empty.")

In [ ]:
# --- Cleanup ---
snowflake_connection.close()
if INCLUDE_REDSHIFT and redshift_connection is not None:
    redshift_connection.close()
print("Connections closed.")